# Lab 10 — Métodos de Propagación de Etiquetas
**Grupo 03** | Minería de Datos — Sección 20  
Universidad del Valle de Guatemala

**Integrantes:** Felipe Aguilar, Vianka Castro, Nicolás Concuá, Ricardo Godínez, Fernando Hernández, Fernando Rueda

---

## Fase 2 — Selección y Análisis de Dataset

**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Fuente:** UCI Machine Learning Repository / [Kaggle](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)  
**Algoritmos a aplicar:** Label Propagation & Label Spreading

---
## 2a. Importación del Dataset

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Carga del dataset desde sklearn (fuente: UCI Machine Learning Repository)
data_raw = load_breast_cancer(as_frame=True)

df = data_raw.frame

# La columna target viene como 0=malignant, 1=benign — agregamos etiqueta textual
df['diagnosis'] = df['target'].map({0: 'malignant', 1: 'benign'})

print('Dataset cargado exitosamente.')
print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')

In [ ]:
# Vista de las primeras filas
df.head()

In [ ]:
# Nombres de todas las columnas
print('Columnas del dataset:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Descripción general del dataset
print(data_raw.DESCR)

---
## 2b. Análisis Exploratorio de Datos (EDA)

### Tipos de variables y dimensionalidad

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Tipos de datos por columna
type_summary = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notna().sum(),
    'null': df.isna().sum(),
    'unique_vals': df.nunique()
})

print('=== Resumen de tipos de variables ===')
print(type_summary.to_string())
print(f'\nTotal features numéricas : {df.select_dtypes(include=np.number).shape[1]}')
print(f'Total features categóricas: {df.select_dtypes(include="object").shape[1]}')

In [ ]:
# Features que se usarán (sin target ni diagnosis)
feature_cols = data_raw.feature_names.tolist()
X = df[feature_cols]
y = df['target']

print(f'Dimensionalidad del espacio de features: {X.shape[1]} dimensiones')
print(f'Número de muestras                      : {X.shape[0]}')
print(f'\nGrupos de features (10 por grupo):')
groups = ['mean', 'error (se)', 'worst']
for g in groups:
    cols = [c for c in feature_cols if g.split()[0] in c]
    print(f'  {g:12s}: {len(cols)} features → {cols[:3]}...')

### Balance de clases

In [ ]:
class_counts = df['diagnosis'].value_counts()
class_pct    = df['diagnosis'].value_counts(normalize=True) * 100

print('=== Balance de clases ===')
for cls in class_counts.index:
    print(f'  {cls:12s}: {class_counts[cls]:4d} muestras ({class_pct[cls]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Countplot
sns.countplot(data=df, x='diagnosis', ax=axes[0],
              order=['malignant', 'benign'],
              palette={'malignant': '#e74c3c', 'benign': '#2ecc71'})
axes[0].set_title('Distribución de clases (conteo)')
axes[0].set_xlabel('Diagnóstico')
axes[0].set_ylabel('Cantidad')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values,
            labels=class_counts.index,
            autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'],
            startangle=90,
            explode=(0.04, 0))
axes[1].set_title('Proporción de clases')

plt.suptitle('Balance de clases — Breast Cancer Wisconsin', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Estadísticas descriptivas

In [ ]:
desc = X.describe().T
desc['cv'] = desc['std'] / desc['mean']  # coeficiente de variación
desc = desc[['mean', 'std', 'cv', 'min', '25%', '50%', '75%', 'max']]
desc.columns = ['Media', 'Std', 'CV', 'Mín', 'Q1', 'Mediana', 'Q3', 'Máx']

print('=== Estadísticas descriptivas (features) ===')
desc.round(4)

### Distribución de features por clase (mean features)

In [ ]:
mean_features = [c for c in feature_cols if 'mean' in c]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

palette = {'malignant': '#e74c3c', 'benign': '#2ecc71'}

for i, feat in enumerate(mean_features):
    sns.histplot(data=df, x=feat, hue='diagnosis',
                 ax=axes[i], kde=True,
                 palette=palette, alpha=0.6, bins=30)
    axes[i].set_title(feat.replace(' mean', ''), fontsize=10)
    axes[i].set_xlabel('')
    if i != 0:
        axes[i].get_legend().remove()

plt.suptitle('Distribución de features (mean) por clase', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Boxplots por clase (mean features)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, feat in enumerate(mean_features):
    sns.boxplot(data=df, x='diagnosis', y=feat,
                ax=axes[i], palette=palette,
                order=['malignant', 'benign'])
    axes[i].set_title(feat.replace(' mean', ''), fontsize=10)
    axes[i].set_xlabel('')

plt.suptitle('Boxplots por clase (mean features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Matriz de correlación

In [ ]:
corr_matrix = X[mean_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.5,
            xticklabels=[c.replace(' mean', '') for c in mean_features],
            yticklabels=[c.replace(' mean', '') for c in mean_features])
ax.set_title('Matriz de correlación — mean features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Pares altamente correlacionados (|r| > 0.9)
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.9:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r, 3)))

print(f'Pares con |r| > 0.90 (riesgo de multicolinealidad):')
for a, b, r in high_corr:
    print(f'  {a}  ↔  {b}  →  r = {r}')

### Detección de outliers (IQR)

In [ ]:
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

outlier_mask = ((X < (Q1 - 1.5 * IQR)) | (X > (Q3 + 1.5 * IQR)))
outlier_counts = outlier_mask.sum().sort_values(ascending=False)

print('=== Outliers por feature (criterio IQR) ===')
print(outlier_counts[outlier_counts > 0].to_string())
print(f'\nMuestras con al menos 1 outlier: {outlier_mask.any(axis=1).sum()} / {len(df)}')

# Visualización de las 5 features con más outliers
top5 = outlier_counts.head(5).index.tolist()
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, feat in enumerate(top5):
    sns.boxplot(data=df, y=feat, x='diagnosis',
                ax=axes[i], palette=palette,
                order=['malignant', 'benign'])
    axes[i].set_title(feat[:22], fontsize=9)
    axes[i].set_xlabel('')

plt.suptitle('Top 5 features con más outliers (IQR)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Scatter plot 2D — primeras 2 features (separabilidad visual)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter con primeras 2 features originales
colors = df['target'].map({0: '#e74c3c', 1: '#2ecc71'})
axes[0].scatter(X.iloc[:, 0], X.iloc[:, 1], c=colors, alpha=0.6, edgecolors='k', linewidths=0.3)
axes[0].set_xlabel(feature_cols[0])
axes[0].set_ylabel(feature_cols[1])
axes[0].set_title('Primeras 2 features originales')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='malignant'),
                   Patch(facecolor='#2ecc71', label='benign')]
axes[0].legend(handles=legend_elements)

# Scatter PCA
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6, edgecolors='k', linewidths=0.3)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
axes[1].set_title('Proyección PCA (2 componentes)')
axes[1].legend(handles=legend_elements)

plt.suptitle('Separabilidad de clases en el espacio de features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Varianza explicada: PC1={pca.explained_variance_ratio_[0]*100:.2f}%, '
      f'PC2={pca.explained_variance_ratio_[1]*100:.2f}%, '
      f'Total={sum(pca.explained_variance_ratio_)*100:.2f}%')

### Valores faltantes

In [ ]:
missing = X.isna().sum()
print('=== Valores faltantes por feature ===')
if missing.sum() == 0:
    print('  No se encontraron valores faltantes en ninguna feature. ✓')
else:
    print(missing[missing > 0])

---
## 2c–2d. Preprocesamiento de Datos

Los algoritmos de Label Propagation y Label Spreading construyen la matriz de afinidad W a partir de distancias euclidianas. Variables con rangos muy distintos dominarían esa distancia, distorsionando la estructura del grafo. Por ello se aplica **estandarización z-score** (media 0, std 1) tal como recomienda la investigación teórica del grupo.

Adicionalmente, como todos los datos del Breast Cancer Wisconsin son variables numéricas continuas, **no se requiere one-hot encoding**. Se documenta esta decisión explícitamente.

### Verificación de variables categóricas

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

cat_cols = X.select_dtypes(include='object').columns.tolist()

if len(cat_cols) == 0:
    print('No existen variables categóricas en el espacio de features. ✓')
    print('One-hot encoding no requerido.')
else:
    print(f'Variables categóricas encontradas: {cat_cols}')

### Estandarización z-score (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

print('=== Verificación post-estandarización ===')
stats_post = X_scaled_df.describe().T[['mean', 'std', 'min', 'max']]
stats_post.columns = ['Media', 'Std', 'Mín', 'Máx']
print(stats_post.round(4).to_string())
print(f'\nMedia global : {X_scaled_df.values.mean():.6f}  (esperado ≈ 0)')
print(f'Std global   : {X_scaled_df.values.std():.6f}   (esperado ≈ 1)')

### Comparación de distribuciones antes y después de estandarizar

In [ ]:
# Comparar 6 mean features representativas
sample_feats = mean_features[:6]

fig, axes = plt.subplots(2, 6, figsize=(22, 7))

for i, feat in enumerate(sample_feats):
    # Antes
    sns.histplot(X[feat], ax=axes[0, i], kde=True, color='#3498db', bins=30, alpha=0.7)
    axes[0, i].set_title(feat.replace(' mean', ''), fontsize=9)
    axes[0, i].set_xlabel('')
    if i == 0:
        axes[0, i].set_ylabel('Antes (escala original)', fontsize=9)

    # Después
    sns.histplot(X_scaled_df[feat], ax=axes[1, i], kde=True, color='#e67e22', bins=30, alpha=0.7)
    axes[1, i].set_xlabel('')
    if i == 0:
        axes[1, i].set_ylabel('Después (z-score)', fontsize=9)

plt.suptitle('Distribución de features: antes vs. después de estandarización', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Impacto de la estandarización en la distancia euclidiana

In [ ]:
from sklearn.metrics import pairwise_distances

# Muestra aleatoria de 100 puntos para visualizar
rng = np.random.default_rng(42)
idx = rng.choice(len(X), size=100, replace=False)

dist_before = pairwise_distances(X.values[idx], metric='euclidean').flatten()
dist_after  = pairwise_distances(X_scaled[idx], metric='euclidean').flatten()
dist_before = dist_before[dist_before > 0]
dist_after  = dist_after[dist_after > 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(dist_before, bins=50, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].set_title('Distancias euclidianas — sin estandarizar')
axes[0].set_xlabel('Distancia')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(dist_after, bins=50, color='#e67e22', alpha=0.8, edgecolor='white')
axes[1].set_title('Distancias euclidianas — estandarizadas')
axes[1].set_xlabel('Distancia')

plt.suptitle('Efecto de la estandarización sobre las distancias (n=100 muestras)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Sin estandarizar  — media: {dist_before.mean():.2f}, rango: [{dist_before.min():.2f}, {dist_before.max():.2f}]')
print(f'Con estandarizar  — media: {dist_after.mean():.2f},  rango: [{dist_after.min():.2f}, {dist_after.max():.2f}]')

### Preparación del conjunto semi-supervisado

Label Propagation y Label Spreading requieren que la mayoría de muestras estén **sin etiquetar** (`y = -1` en scikit-learn). Se simula un escenario realista donde solo el **20% de las muestras tienen etiqueta conocida**, manteniendo el balance de clases mediante estratificación.

In [ ]:
LABELED_RATIO = 0.20  # 20% etiquetado, 80% no etiquetado
RANDOM_STATE  = 42

# Split estratificado: 20% labeled, 80% unlabeled
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y.values,
    test_size=0.80,
    stratify=y.values,
    random_state=RANDOM_STATE
)

# Construir vector y_semi: etiquetas conocidas + -1 para no etiquetados
y_semi = np.concatenate([y_train, np.full(len(y_test), -1)])
X_semi = np.vstack([X_train, X_test])

print('=== Configuración semi-supervisada ===')
print(f'Total muestras         : {len(y_semi)}')
print(f'Etiquetadas   (20%)    : {(y_semi != -1).sum()} '
      f'[{(y_semi[y_semi != -1] == 0).sum()} malignas, '
      f'{(y_semi[y_semi != -1] == 1).sum()} benignas]')
print(f'No etiquetadas (80%)   : {(y_semi == -1).sum()}')

In [ ]:
# Visualización del conjunto semi-supervisado (proyección PCA)
pca2 = PCA(n_components=2, random_state=42)
X_semi_pca = pca2.fit_transform(X_semi)

labeled_mask   = y_semi != -1
unlabeled_mask = y_semi == -1

fig, ax = plt.subplots(figsize=(9, 6))

# Puntos no etiquetados
ax.scatter(X_semi_pca[unlabeled_mask, 0], X_semi_pca[unlabeled_mask, 1],
           c='lightgray', alpha=0.5, s=25, label='No etiquetado', zorder=1)

# Puntos etiquetados
colors_labeled = np.where(y_semi[labeled_mask] == 0, '#e74c3c', '#2ecc71')
ax.scatter(X_semi_pca[labeled_mask, 0], X_semi_pca[labeled_mask, 1],
           c=colors_labeled, s=60, edgecolors='black', linewidths=0.5,
           label='Etiquetado', zorder=2)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightgray', label=f'No etiquetado ({unlabeled_mask.sum()})'),
    Patch(facecolor='#e74c3c',   label=f'Maligno etiquetado ({(y_semi[labeled_mask]==0).sum()})'),
    Patch(facecolor='#2ecc71',   label=f'Benigno etiquetado ({(y_semi[labeled_mask]==1).sum()})')
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Conjunto semi-supervisado — 20% etiquetado (proyección PCA)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Resumen del dataset preprocesado

In [ ]:
print('=' * 50)
print('  RESUMEN — Dataset preprocesado (Fase 2)')
print('=' * 50)
print(f'  Dataset          : Breast Cancer Wisconsin')
print(f'  Muestras totales : {X_semi.shape[0]}')
print(f'  Features         : {X_semi.shape[1]} (todas numéricas)')
print(f'  Clases           : 2 (0=malignant, 1=benign)')
print(f'  Valores faltantes: 0')
print(f'  Encoding         : No requerido (sin variables categóricas)')
print(f'  Escalado         : StandardScaler (z-score)')
print(f'  Etiquetadas      : {(y_semi != -1).sum()} ({LABELED_RATIO*100:.0f}%)')
print(f'  No etiquetadas   : {(y_semi == -1).sum()} ({(1-LABELED_RATIO)*100:.0f}%)')
print('=' * 50)
print('  Listo para Label Propagation / Label Spreading ✓')
print('=' * 50)